# 第19章 少ないメモリで鍛える：LoRAと量子化

本体には指一本触れず、小さな「付せん」だけを学習する

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/19-lora.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：LoRA層をゼロから実装してQwenに取り付け、学習するパラメータを1%以下にしてボンゴ先生化する。さらにPEFTとQLoRA（4ビット量子化＋LoRA）で、無料GPUに大きめのモデルを載せる
- **身につく数学・理論**：行列のランクと低ランク近似 ／ LoRA：重みの変化を小さな2つの行列の積で表す ／ 学習するパラメータ数とメモリの節約 ／ 量子化（absmax・int8・4ビット）と誤差 ／ QLoRA：4ビットの本体＋LoRA ／ アダプタの保存・合体（マージ）
- **新しく使うPython・PyTorch**：`requires_grad_(False)` で凍結する ／ モジュールの差し替え（`attn.q_proj = …`） ／ `peft` の `LoraConfig` と `PeftModel` ／ `BitsAndBytesConfig`（4ビット読み込み）

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（GPU：T4） |
| 所要時間目安 | 2時間 |
| 難易度 | ★★★★☆ |

> **⚠️ GPUランタイム設定**：メニューの「ランタイム」→「ランタイムのタイプを変更」→「**T4 GPU**」を選んでから上から順に実行してください。
> 無料枠の混雑・上限でGPUに接続できない場合は、時間をおくか [Kaggle Notebooks](https://sakunokyumei.github.io/llm-research-textbook/a-setup.html) を使うか、各章のCPU短縮手順で進められます。接続切断に備え、学習済みモデルはこまめにGoogleドライブへ保存しましょう。

> **🎯 この章でつくるもの**
>
> 第18章では、0.5Bのモデルの全パラメータを学習させるのに約10GBのメモリを使いました。この方法では、7Bのモデルは無料のGPUに乗りません。この章では、モデル本体の重みを**凍結**（変えない）したまま、ほんの少しの追加パラメータだけを学習する **LoRA** を、まず自分の手で実装します。さらに、モデル本体を4ビットに縮める **量子化** と組み合わせた **QLoRA** で、無料のT4でも大きめのモデルを鍛えられるようにします。

> **🔗 これまでの復習とつながり：第1章・第6章・第18章とのつながり**
>
> - **第1章・第15章（VRAMの見積もり）**：学習時のメモリは「重み＋勾配＋Adamの記録 ≒ 16バイト × パラメータ数」で決まり、T4 GPUの16GB上限が大きな壁になることを学びました。
> - **第6章（行列のかけ算の形）**：形 `(m, k)` の行列と `(k, n)` の行列を掛けると `(m, n)` になるルールを使います。LoRAの $BA$ と `x @ A.T @ B.T` はまさにこの形の組み合わせです。
> - **第18章（全パラメータSFT）**：損失マスク `-100` とパディングで作ったボンゴ先生データを、そのままこの章のLoRA学習に再利用します！
>
> この章では、**「なぜ低ランク行列 $BA$ でパラメータが1%以下に縮むのか」「なぜ $B=0$・$A$ はランダムに初期化するのか」「量子化の丸め誤差はどう計算されるのか」** を手計算と実装の両面からマスターします。

**ミナト**：本体を変えずに鍛えるって、どういうことですか？ 筋トレしないでマッチョになる、みたいな……？

**ボンゴ先生**：いい例えだ（いや、そうでもない）。教科書に書き込みをする代わりに、**付せん**を貼るイメージだ。教科書（元のモデル）はきれいなまま。付せん（LoRA）に「ここはこう読み替える」と書いておく。付せんは小さいから、何枚作っても軽い。しかも剥がせば元通り。用途ごとに付せんを取り替えることもできる。

## なぜ全部を学習すると重いのか

第15章で計算したとおり、AdamWで学習するときのメモリは、だいたい「16バイト × パラメータ数」でした。

| モデル | 推論（float16） | 全パラメータの学習 |
|---|---|---|
| 0.5B | 約1GB | 約8GB |
| 1.5B | 約3GB | 約24GB（T4に乗らない） |
| 7B | 約14GB | 約112GB（T4が7枚以上） |

学習のメモリの大半は、**勾配**と**Adamの記録**（慣性と歩幅）です。どちらも「学習するパラメータの数」に比例します。だったら、**学習するパラメータを減らせば**いいのです。

#### 📝 確認問題（`ch19-q01` · Lv.2 · 全パラメータ学習に必要なVRAMの概算（16バイト × パラメータ数））

AdamWで全パラメータをfloat32で学習すると、おおよそ「16バイト × パラメータ数」（重み4B＋勾配4B＋Adam記録8B）のVRAMが必要です。**1.5B（15億）パラメータ**のモデルを全パラメータ学習するとき、目安のメモリは約何GBになりますか？（整数で答えてください）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：$1\text{B} = 10^9$ 個のパラメータに1バイトずつ使うと約 $1\text{GB}$ になります。パラメータ数（B）と1パラメータあたりのバイト数の積で概算できます。
- **ヒント 2**：パラメータ数 `1.5`（B）に、全パラメータ学習の目安 `16` バイトを掛けてGB単位の見積もりを求めましょう。
- **正解**：`24` GB

**考え方**：1.5B × 16バイト = 24GB となり、無料ColabのT4 GPU（VRAM 16GB）には収まりません。勾配とAdam記録の12バイト分が学習パラメータ数に比例して膨らむためです。

1. 1パラメータあたりの学習メモリ：重み4B + 勾配4B + Adam記録（慣性4B + 歩幅4B）= 16バイト。
1. 1.5B × 16バイト = 24GB（T4の16GBを超過）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/19-lora.html#ex-ch19-q01

</details>

## 行列のランク：情報は意外と少ない

LoRAのアイデアの元になる、**ランク**（階数）という考え方を見ておきましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import torch
torch.manual_seed(0)
A = torch.randn(6, 2)
B = torch.randn(2, 6)
W = A @ B                       # 6×2 と 2×6 の積 → 6×6の行列
print(W.shape, "のランク：", torch.linalg.matrix_rank(W).item())
print("Wの数の個数：", W.numel(), " AとBの数の個数：", A.numel() + B.numel())

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
torch.Size([6, 6]) のランク： 2
Wの数の個数： 36  AとBの数の個数： 24
```

</details>

#### 📐 数学のポイント：ランクとLoRA（$W' = W + \frac{\alpha}{r}BA$）：7ステップで理解する

**① 解決したい問題**：巨大な重み行列 $W$（たとえば 7B モデルにある $4096 \times 4096 = 1{,}677\text{万}$ 個の重み）を直接更新すると、勾配とAdamの記録だけでVRAMがパンクしてしまう。元の重み $W$ を凍結したまま、ほんのわずかなパラメータ数で表現力豊かな調整 $\Delta W$ を加えたい。

**② 素朴な方法では困る理由**：全要素を直接学習すると 16バイト × パラメータ数でメモリ不足になる。かといって最後の1層だけを学習するだけでは、モデル全体の口調や推論パターンを十分に変えられない。

**③ 発想（低ランク近似）**：6×6の行列 `W` には36個の数が並んでいますが、6×2と2×6の行列の積で作ったので、本当は**2次元ぶんの情報（ランク2）**しか持っていません。2021年にMicrosoftの研究者が提案した **LoRA**（Low-Rank Adaptation）は、「ファインチューニングでの重みの**変化 $\Delta W$** は低いランク $r$ で十分に表せる」という発見に基づき、変化ぶんだけを2つの細い行列 $B$ と $A$ の積に分解します。

**④ 記号表と数式**：

$$W' = W + \Delta W = W + \frac{\alpha}{r} B A$$

| 記号 | 形（shape） | 凍結／学習 | 意味 |
|---|---|---|---|
| $W$ | $(d_{\text{out}},\; d_{\text{in}})$ | **凍結**（更新しない） | 事前学習済みの元の重み行列 |
| $A$ | $(r,\; d_{\text{in}})$ | **学習する** | 入力を細い $r$ 次元へ圧縮（ランダム正規分布で初期化） |
| $B$ | $(d_{\text{out}},\; r)$ | **学習する** | $r$ 次元を元の出力次元へ展開（**すべて `0` で初期化**） |
| $r$ | スカラー整数（例：4, 8, 16） | ハイパーパラメータ | ランク（通り道の細さ・ボトルネックの幅） |
| $\alpha$ | スカラー（例：16, 32） | ハイパーパラメータ | 変化の効き具合を調整する倍率（$\frac{\alpha}{r}$ を掛ける） |

**⑤ 具体的な手計算でパラメータ削減量を確かめよう**：
一般の $d_{\text{out}} \times d_{\text{in}}$ の層で、元のパラメータ数は $d_{\text{out}} d_{\text{in}}$ 個、LoRAは $r(d_{\text{in}} + d_{\text{out}})$ 個です。
- **7Bクラスの層（$4096 \times 4096$、$r = 8$）**：
  - 元の重み $W$：$4096 \times 4096 = \mathbf{16{,}777{,}216}$ 個（約 16.78M）
  - LoRA（$A$ と $B$）：$8 \times (4096 + 4096) = \mathbf{65{,}536}$ 個（約 65.5K）
  - 割合：$65{,}536 \div 16{,}777{,}216 \approx \mathbf{0.39\%}$！（99.6%削減！）
- **この章のQwen2.5-0.5B（$896 \times 896$、$r = 8$）**：
  - 元の重み $W$：$896 \times 896 = 802{,}816$ 個 → LoRA：$8 \times (896 + 896) = \mathbf{14{,}336}$ 個（約 **1.79%**）

**⑥ なぜ「$B = 0$、$A =$ ランダム正規分布」で初期化するのか？**
1. **ステップ0で元のモデルを完全に守る**：$B = 0$ なら $\Delta W = \frac{\alpha}{r}BA = 0$ となり、学習開始瞬間の出力は事前学習済みモデルと**100%一致**します（最初から元の能力を壊さない！）。
2. **なぜ $A$ まで `0` にしてはいけないのか？**：連鎖律より、$B$ の勾配は $A$ に比例し、$A$ の勾配は $B$ に比例します。もし**両方とも `0`** にすると、$A$・$B$ 両方の勾配がずっと `0` のままになって学習が一歩も進みません！ $A$ をランダム（非ゼロ）にしておくことで、ステップ1から $B$ に非ゼロの勾配が流れて動き出し、ステップ2以降は $A$ も一緒に学習されます。

**⑦ 理解のための最小Pythonコード**：

```python no-run
d_in, d_out, r = 4096, 4096, 8
orig = d_in * d_out                 # 16,777,216
lora = r * (d_in + d_out)           # 65,536
print(f"{lora / orig:.2%}")         # 0.39%
```

#### 📝 確認問題（`ch19-q02` · Lv.2 · LoRAの学習パラメータ数 r(d_in + d_out) の手計算）

$1024 \times 1024$ の全結合層（バイアス除く）に、ランク **$r = 4$** のLoRAを取り付けると、学習するパラメータ数（$A$ と $B$ の合計）はいくつになりますか？

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：LoRAでは元の行列 $W$（$d_{\text{out}} \times d_{\text{in}}$）に対し、$A$（$r \times d_{\text{in}}$）と $B$（$d_{\text{out}} \times r$）の2行列だけを学習します。
- **ヒント 2**：合計パラメータ数の公式 $r(d_{\text{in}} + d_{\text{out}})$ に $r = 4, d_{\text{in}} = 1024, d_{\text{out}} = 1024$ を当てはめて計算しましょう。
- **正解**：`8192` 個

**考え方**：$A$ に $4 \times 1024 = 4096$ 個、$B$ に $1024 \times 4 = 4096$ 個で合計 $8192$ 個。元の $1024^2 = 1{,}048{,}576$ 個のわずか約0.78%です。

1. $A$ の要素数：$r \times d_{\text{in}} = 4 \times 1024 = 4096$。
1. $B$ の要素数：$d_{\text{out}} \times r = 1024 \times 4 = 4096$。
1. 合計：$4096 + 4096 = 8192$ 個。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/19-lora.html#ex-ch19-q02

</details>

#### 📝 確認問題（`ch19-q03` · Lv.2 · LoRAで B=0、A=ランダム正規分布に初期化する理由）

LoRAの式 $W' = W + \frac{\alpha}{r}BA$ で、学習開始時に行列 **$B$ をすべて0** に、**$A$ をランダムな正規分布** に初期化する理由として正しいものはどれですか？

- **a.** 要素がすべて0の行列はPyTorchが疎テンソルとして圧縮するため、$B$ のぶんGPUメモリを一切消費しないから
- **b.** $B$ を0で初期化しておくと学習中も $B$ がずっと0のまま固定され、$A$ だけを更新する片側学習になるから
- **c.** $A$ と $B$ を両方ランダム初期化すると積 $BA$ のランクが必ず1に低下し、表現力が大きく損なわれるから
- **d.** 開始時に $BA=0$ で元の出力を完全保持でき、かつ $A \neq 0$ なので1歩目から $B$ に非ゼロ勾配が流れるから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：①ステップ0で事前学習済みモデルの出力を壊さない条件（$\Delta W = 0$）と、②ステップ1で勾配が消えない条件、の2つを同時に満たす必要があります。
- **ヒント 2**：$B = 0$ なら $BA = 0$ です。そのとき連鎖律で $B$ の勾配が $A$ に比例することに着目しましょう。
- **正解**：`d`

**考え方**：$B=0$ なら $BA=0$ なので初期出力は元のモデルと完全一致します。さらに $A \neq 0$ なので逆伝播で $B$ の勾配（$A$ に比例）が0にならず、1ステップ目から確実に学習が動き出します。

1. ステップ0：$B = 0 \implies W' = W + \frac{\alpha}{r} \cdot 0 \cdot A = W$（元の能力を100%保持）。
1. ステップ1の逆伝播：$B$ の勾配は入力 $x A^T$ に比例するため、$A \neq 0$ なら $B$ が更新される。
1. ステップ2以降：$B \neq 0$ になったので、今度は $A$ の勾配（$B^T$ に比例）も0でなくなり両方学習される。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/19-lora.html#ex-ch19-q03

</details>

*(この節の追加問題 `ch19-q04` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/19-lora.html で自動採点・復習できます)*

## LoRA層を自分で作る

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=16):
        super().__init__()
        self.base = base
        self.base.requires_grad_(False)                       # 本体は凍結
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))   # 最初は0
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale

base = nn.Linear(896, 896)
lora = LoRALinear(base)
x = torch.randn(2, 896)
print("最初は元の層と同じ？", torch.allclose(base(x), lora(x)))
trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)
print(f"学習するパラメータ：{trainable:,} / 元の層：{base.weight.numel() + base.bias.numel():,}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
最初は元の層と同じ？ True
学習するパラメータ：14,336 / 元の層：803,712
```

</details>

> **🐍 Python・PyTorchのポイント：requires_grad_(False) と nn.Parameter**
>
> `requires_grad_(False)` は「このパラメータの勾配は計算しなくていい（学習しない）」という設定です。勾配もAdamの記録も作られないので、メモリを食いません。これを **凍結**（freeze）と言います。<br>
> `nn.Parameter(テンソル)` は、そのテンソルを「このモデルの学習するパラメータ」として登録します。こうしておくと、`model.parameters()` に含まれ、オプティマイザが更新してくれます。<br>
> `def __init__(self, base: nn.Linear, …)` の `: nn.Linear` は **型ヒント**（「ここには全結合層が入ります」というメモ）で、動きは変わりません。第23章でもう一度出てきます。

<details><summary><b>基礎の補足：`x @ self.A.T @ self.B.T` のテンソル形状トレース（なぜ `.T` が要るのか？）</b>（クリックで開く）</summary>

PyTorchの `nn.Linear(in_features, out_features)` は重みを `(out_features, in_features)` で持ち、入力 `x`（形 `(batch, in_features)`）に右から転置を掛けます。LoRAの2つの行列も同じ流儀で `A: (r, d_in)`、`B: (d_out, r)` と定義しているため、左から順に次のように形が変化します（例：`batch=2, d_in=896, r=8, d_out=896`）。

| ステップ | 式 | テンソルの形 | 役割 |
|---|---|---|---|
| 入力 | `x` | `(2, 896)` | バッチ2件 × 896次元 |
| 圧縮 | `x @ self.A.T` | `(2, 896) @ (896, 8) → (2, 8)` | 細いボトルネック（ランク8）へギュッと圧縮 |
| 展開 | `(x @ self.A.T) @ self.B.T` | `(2, 8) @ (8, 896) → (2, 896)` | 出力896次元へ戻して `self.base(x)` に足す |

`B @ A` を先に計算して `(896, 896)` の巨大行列を作らず、左から `(x @ A.T) @ B.T` と細い `(2, 8)` を経由するのが省メモリ＆高速のコツです！

</details>

#### 📝 確認問題（`ch19-q05` · Lv.2 · 長方形の全結合層に対するLoRAパラメータ数と割合の予測）

入力512次元・出力1024次元の全結合層にランク `r = 8` のLoRAを付けたときの、LoRAパラメータ数と元の層に対する割合（%）を計算するコードです。実行すると何が表示されますか？

```python
d_in, d_out, r = 512, 1024, 8
orig = d_in * d_out
lora = r * (d_in + d_out)
print(lora, round(lora / orig * 100, 2))
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：元のパラメータ数は `d_in * d_out`、LoRAパラメータ数は `r * (d_in + d_out)` です。割合（%）は `lora / orig * 100` を小数第2位へ丸めます。
- **ヒント 2**：`d_in + d_out = 1536` に `r = 8` を掛けて `lora` を求め、`orig = 512 * 1024 = 524288` に対するパーセントを計算しましょう。
- **期待される出力**：`12288 2.34`

**考え方**：長方形の層でも $A$ が `(r, d_in)`、$B$ が `(d_out, r)` なので合計は `r * (d_in + d_out) = 12288` となり、元の `524288` 個の `2.34%` です。

1. `orig = 512 * 1024 = 524288`。
1. `lora = 8 * (512 + 1024) = 8 * 1536 = 12288`。
1. `12288 / 524288 * 100 = 2.34375` → 小数第2位で丸めて `2.34`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/19-lora.html#ex-ch19-q05

</details>

#### 📝 確認問題（`ch19-q06` · Lv.2 · LoRALinear の順伝播 x @ A.T @ B.T のテンソル形状変化）

自作した `LoRALinear` で、入力 `x` の形が `(2, 896)`、`self.A` が `(8, 896)`、`self.B` が `(896, 8)` のとき、`x @ self.A.T @ self.B.T` を左から順に計算した中間テンソルと最終出力の形の正しい組み合わせはどれですか？

- **a.** 転置 `.T` を外して `x @ self.A @ self.B` と書いても内側の次元が一致し、同じ `(2, 896)` になる
- **b.** 結合順序の関係で必ず `A` と `B` を先に掛けて `(896, 896)` を作ってからでないと計算できない
- **c.** `x @ A.T` で `(896, 896)` の正方行列へ広がり、次に `@ B.T` で `(8, 8)` の小行列へ圧縮される
- **d.** `x @ A.T` で `(2, 8)` に圧縮され、次に `@ B.T` で `(2, 896)` に戻って `base(x)` と同じ形になる

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：行列積 `(m, k) @ (k, n) -> (m, n)` のルールを左から順番に適用します。まず `self.A.T` と `self.B.T` の形を確認しましょう。
- **ヒント 2**：`x` は `(2, 896)`、`A.T` は `(896, 8)`、`B.T` は `(8, 896)` です。最初の積の中間形状と最終形状を順に追ってみましょう。
- **正解**：`d`

**考え方**：入力 `(2, 896)` を `A.T` `(896, 8)` で細い8次元ボトルネック `(2, 8)` に押し込み、`B.T` `(8, 896)` で出力次元 `(2, 896)` へ展開して `self.base(x)` に足し合わせます。

1. `x`: `(2, 896)`, `A.T`: `(896, 8)` → `x @ A.T` は `(2, 8)`。
1. `(2, 8)` と `B.T`: `(8, 896)` → `(x @ A.T) @ B.T` は `(2, 896)`。
1. `self.base(x)` も `(2, 896)` なので要素ごとに足し算できる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/19-lora.html#ex-ch19-q06

</details>

*(この節の追加問題 `ch19-q07` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/19-lora.html で自動採点・復習できます)*

## Qwenに付せんを貼る

> **✨ コツ：実行環境とリソースの目安（後半のQLoRAはGPU必須）**
>
> - **環境**：自作LoRAとPEFT節（0.5B）は無料GPU（T4）推奨（VRAM約3〜4GB、学習約1分）。後半の **QLoRA節（1.5B・4ビット量子化）は `bitsandbytes` がCUDAを必要とするためGPU必須**（T4でVRAM約4GB、約1〜2分）です。
> - **ダウンロードと保存**：0.5B（約1GB）＋QLoRA節で1.5B（約3.1GB）。保存するLoRAアダプタは約35MBで済みます。
> - **GPUが使えない日の代替**：行列のランク・自作 `LoRALinear`・absmax量子化の手計算セルは**CPUで数秒**で動きます。0.5Bの自作LoRA学習もCPUで `train_rows[:16]`・`epochs=1` に絞れば動作確認できます。QLoRA節だけはGPU確保後（またはKaggle T4）に実行してください。
> - **ライブラリ互換性**：`dtype=` は `transformers >= 4.56`、`device_map="auto"` には `accelerate` が必要です（推奨環境は `requirements-book.txt`）。

自作のLoRA層を、Qwen2.5-0.5B-Instructの全Attentionの `q_proj` と `v_proj`（QueryとValueを作る全結合層）に取り付けます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
!wget -nv https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/bongo_sft.jsonl
!wget -nv https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/bongo_test.jsonl
import json, os
for fname in ("bongo_sft.jsonl", "bongo_test.jsonl"):
    if not os.path.exists(fname):
        raise RuntimeError(f"{fname} のダウンロードに失敗しました。ネットワーク接続を確認して再実行するか、付録E（エラー辞典）を参照してください。")
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
name = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).to(device)
model.requires_grad_(False)                       # まず全部凍結

for layer in model.model.layers:                  # 全層のAttentionに付せんを貼る
    attn = layer.self_attn
    attn.q_proj = LoRALinear(attn.q_proj).to(device)
    attn.v_proj = LoRALinear(attn.v_proj).to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"全パラメータ {total:,} のうち、学習するのは {trainable:,}（{trainable / total:.2%}）")

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
全パラメータ 494,573,440 のうち、学習するのは 540,672（0.11%）
```

</details>

学習するのは、全体の**0.1%ほど**だけになりました。

`attn.q_proj = LoRALinear(attn.q_proj)` の1行で、なぜモデルの計算が変わるのでしょう？ Qwenの中では、Attentionの計算のときに `self.q_proj(x)` と、**その名前の部品**を呼び出しています。だから、`q_proj` という名前の部品を、元の層を包んだLoRA層に差し替えれば、次からはLoRA層が呼ばれるのです。部品を差し替えるだけで、Qwenのコードは1行も書き換えていません。

学習は、第18章のデータ作りと学習ループをそのまま使います（自分で書いたものを再利用！）。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import random
train_rows = [json.loads(l) for l in open("bongo_sft.jsonl", encoding="utf-8")]
test_rows = [json.loads(l) for l in open("bongo_test.jsonl", encoding="utf-8")]

def build_example(question, answer):
    prompt = tok.apply_chat_template([{"role": "user", "content": question}],
                                     add_generation_prompt=True, tokenize=False)
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    a = tok(answer + "<|im_end|>\n", add_special_tokens=False)["input_ids"]
    return {"input_ids": p + a, "labels": [-100] * len(p) + a}

def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    pad = lambda xs, v: [x + [v] * (L - len(x)) for x in xs]
    return {"input_ids": torch.tensor(pad([b["input_ids"] for b in batch], tok.pad_token_id)).to(device),
            "labels": torch.tensor(pad([b["labels"] for b in batch], -100)).to(device),
            "attention_mask": torch.tensor(pad([[1] * len(b["input_ids"]) for b in batch], 0)).to(device)}

examples = [build_example(r["question"], r["answer"]) for r in train_rows]
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
random.seed(0); torch.manual_seed(0)
model.train()
for epoch in range(3):
    random.shuffle(examples)
    total = 0.0
    n_steps = 0
    for i in range(0, len(examples), 4):
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
            loss = model(**collate(examples[i:i + 4])).loss
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        total += loss.item()
        n_steps += 1
    print(f"エポック {epoch + 1}：平均損失 {total / n_steps:.3f}")
model.eval()

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
エポック 1：平均損失 2.873
エポック 2：平均損失 1.792
エポック 3：平均損失 1.415
```

</details>

> **✨ コツ：LoRAの学習率は大きめ**
>
> 全パラメータの学習（第18章）では学習率 $10^{-5}$ でしたが、LoRAでは $10^{-4}$〜$10^{-3}$ と大きめにします。学習するのは小さな付せんだけで、しかも $B$ が0から始まるので、ある程度大きく動かさないと効果が出ないからです。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def chat(question, n=150):
    inputs = tok.apply_chat_template([{"role": "user", "content": question}], add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True).to(device)
    out = model.generate(**inputs, max_new_tokens=n, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(chat(test_rows[6]["question"]))

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
いい質問だ！星は自分の中で核融合という反応を起こして、ものすごいエネルギーを光として出しているんだ。何光年も旅してきた光が、今キミの目に届いている。星だけに、ほしい答えだったかな！
```

</details>

付せんだけで、ボンゴ先生になりました。

#### 📝 確認問題（`ch19-q08` · Lv.2 · 本体凍結（requires_grad_(False)）とLoRAの学習率が大きめな理由）

LoRAで本体に `model.requires_grad_(False)` を実行するとVRAMが大幅に節約できる主な理由と、LoRAの学習率を全パラメータSFT（$10^{-5}$）より大きめ（$10^{-4}$〜$10^{-3}$）にする理由の組み合わせとして正しいものはどれですか？

- **a.** 本体を凍結すると計算対象の層数が自動的に半分へ削減され、学習率を上げるほど壊滅的忘却が確実に0になる
- **b.** 凍結した本体には勾配とAdam状態が確保されずメモリが激減し、$B=0$ の小差分を動かすため学習率は大きめにする
- **c.** `requires_grad_(False)` で本体重みがGPUから削除されるためメモリが減り、学習率は毎回ランダムに決める
- **d.** `requires_grad_(False)` は推論速度を約10倍にする最適化設定であり、学習時のVRAM使用量自体は変わらない

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：①全パラメータ学習メモリの約75%を占めていた「勾配とAdamの記録」が凍結後にどうなるか、②$B=0$ から小さい行列だけを育てる際の更新幅、の2点を整理しましょう。
- **ヒント 2**：本体重み自体は順伝播のためGPUに残りますが、勾配・Adam状態が不要になります。また全パラメータSFT（$10^{-5}$）よりLoRAの学習率を1桁ほど大きくする理由を思い出しましょう。
- **正解**：`b`

**考え方**：オプティマイザには `[p for p in model.parameters() if p.requires_grad]`（全体の0.1〜2%のLoRAパラメータのみ）しか渡さないため、勾配・Adam状態がほぼゼロになります。また $B=0$ からの出発ゆえ、学習率は全パラメータSFTより1桁ほど大きめが適します。

1. `requires_grad_(False)` → 本体4.9億パラメータ分の勾配とAdam状態が不要になる。
1. LoRAの `A, B` だけを `AdamW` に渡す（0.11%の54万個だけ）。
1. $B=0$ から変化を育てるため、学習率は `1e-4`〜`1e-3` 程度に設定する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/19-lora.html#ex-ch19-q08

</details>

## 道具を使う：PEFT

自作のLoRAで仕組みが分かったら、実務では Hugging Faceの **PEFT**（Parameter-Efficient Fine-Tuning）ライブラリを使います。どの層に付せんを貼るか、ランクをいくつにするかを指定するだけです。TRLの `SFTTrainer` と組み合わせると、第18章と同じ書き方でLoRA学習ができます。

📦 **準備セル**：ライブラリや教材ファイルを取得します。エラーが出た場合は `-q` を外して詳細を確認してください。

In [ ]:
!pip install -q peft trl

🔎 **実行前に確認**：コードの各行が何を受け取り、何を作るか確認してから ▶ を押してください。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

ds = Dataset.from_list([{"prompt": [{"role": "user", "content": r["question"]}],
                         "completion": [{"role": "assistant", "content": r["answer"]}]} for r in train_rows])
lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                         target_modules="all-linear", task_type="CAUSAL_LM")
base_model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)
config = SFTConfig(output_dir="lora-bongo", num_train_epochs=3, per_device_train_batch_size=4,
                   learning_rate=2e-4, logging_steps=5, report_to="none", save_strategy="no",
                   bf16=False, fp16=torch.cuda.is_available())
trainer = SFTTrainer(model=base_model, args=config, train_dataset=ds,
                     processing_class=tok, peft_config=lora_config)
trainer.model.print_trainable_parameters()
trainer.train()

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497
{'loss': 2.81, 'grad_norm': 1.9, 'learning_rate': 0.000178, 'epoch': 0.33}
...
{'train_runtime': 38.5, 'train_loss': 1.71, 'epoch': 3.0}
```

</details>

`target_modules="all-linear"` は「全部の全結合層に付せんを貼る」指定です。Attentionだけでなく、フィードフォワード層にも貼ると、効果が上がることが多いと言われています。

### 付せんだけを保存する

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
trainer.model.save_pretrained("bongo-lora-adapter")
import os
size = sum(os.path.getsize(os.path.join("bongo-lora-adapter", f)) for f in os.listdir("bongo-lora-adapter"))
print(f"保存したアダプタの大きさ：{size / 1e6:.1f} MB")

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
保存したアダプタの大きさ：35.2 MB
```

</details>

第18章で全体を保存したときは約2GBでした。LoRAでは、付せん（**アダプタ**）だけを保存すればよいので、ずっと小さくて済みます。使うときは、元のモデルを読み込んでから、アダプタを貼り付けます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
from peft import PeftModel
reloaded = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(name), "bongo-lora-adapter")
merged = reloaded.merge_and_unload()      # 付せんを本体に書き込んで、ふつうのモデルに戻す
print(type(merged).__name__)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
Qwen2ForCausalLM
```

</details>

`merge_and_unload()` は、$W' = W + \frac{\alpha}{r}BA$ を実際に計算して本体に書き込み、付せんを外します。こうすると、推論のときに追加の計算が要らなくなります。

#### 📝 確認問題（`ch19-q09` · Lv.2 · LoRAアダプタの軽量保存と merge_and_unload() の仕組み）

PEFTで学習したLoRAアダプタを保存すると約35MB（第18章の丸ごと保存は約2GB）で済み、推論時に `merged = reloaded.merge_and_unload()` を呼ぶことがあります。`merge_and_unload()` が行う処理として正しいものはどれですか？

- **a.** モデル本体を自動で4ビット量子化することで、保存時のファイルサイズを約2GBから35MBまで圧縮する処理
- **b.** 元の重み $W$ に差分 $\frac{\alpha}{r}BA$ を合体し、LoRA層を外して通常速度で推論する
- **c.** 学習済みのLoRAアダプタをすべて破棄し、モデルをファインチューニング前のベースモデルへ完全に巻き戻す処理
- **d.** ファインチューニングに用いたJSONLの対話テキストを、モデルの重みファイルの末尾へそのまま埋め込む処理

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：推論のたびに毎回 `base(x) + (x @ A.T @ B.T) * scale` と2経路を計算するのと、あらかじめ $W' = W + \frac{\alpha}{r}BA$ を1つにまとめる違いを考えましょう。
- **ヒント 2**：線形性の性質により、差分行列を元の重み $W$ へ足し込んでしまえば、追加の計算コストなしで学習後の振る舞いをそのまま再現できます。
- **正解**：`b`

**考え方**：マージ前（`PeftModel`）は推論のたびに `base(x) + (x @ A.T @ B.T) * scale` と2経路の計算が必要ですが、`merge_and_unload()` で $W' = W + \frac{\alpha}{r}BA$ を事前計算してしまえば、推論時の追加計算コストがゼロになります。

1. 保存・配布時：本体は共有し、35MBのアダプタ（$A, B$）だけを保存・切り替えできる。
1. 本番推論時：`merge_and_unload()` で $W' = W + \frac{\alpha}{r}BA$ を本体に書き込む。
1. 結果：クラスも通常の `Qwen2ForCausalLM` に戻り、推論オーバーヘッドが0になる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/19-lora.html#ex-ch19-q09

</details>

## 量子化：数を粗くして小さくする

LoRAで、学習の**勾配とAdamの記録**は小さくなりました。残る大物は、モデル本体の重みそのものです。そこで、重みを少ないビット数で表す **量子化**（quantization）を使います。

一番シンプルな「対称absmax量子化」を、数式と手計算から確認してからコードで動かしましょう。

#### 📐 数学のポイント：対称absmax量子化と逆量子化誤差：7ステップで理解する

**① 解決したい問題**：float32（4バイト）の重み $w$ を、int8（1バイト）や4ビット（0.5バイト）の小さな整数 $q$ に縮めてVRAMを4分の1〜8分の1にしたい。

**② 素朴な方法の失敗**：そのまま小数点以下を切り捨てて整数にすると、LLMの重みのほとんどは `[-0.5, +0.5]` の小さな小数なので、全部 `0` になって情報が消えてしまう。

**③ 発想**：まず重みの中の最大絶対値 $\max_i |w_i|$ を整数上限 $Q_{\max}$（8ビット符号付きなら `127`、4ビット符号付きなら `7`）にぴったり合わせる**縮尺（scale）** $s$ を作り、割って四捨五入する。使うときは $s$ を掛け戻す（逆量子化）。

**④ 記号表と数式**：

$$s = \frac{\max_i |w_i|}{Q_{\max}}, \qquad q_i = \operatorname{round}\!\left(\frac{w_i}{s}\right), \qquad \hat{w}_i = q_i \cdot s, \qquad \text{誤差}_i = |w_i - \hat{w}_i|$$

| 記号 | 意味 | Pythonでの対応 |
|---|---|---|
| $w_i$ | 元のfloat32の重み | `w` |
| $Q_{\max}$ | 整数表現の最大値（8bitなら `127`、4bitなら `7`） | `(1 << (bits - 1)) - 1` |
| $s$ | 縮尺（scale：1目盛りあたりの幅） | `max(abs(x) for x in w) / qmax` |
| $q_i$ | 量子化された整数（保存する値） | `round(x / scale)` |
| $\hat{w}_i$ | 逆量子化で元に戻した重み | `q * scale` |

**⑤ 小さな数字で手計算（4ビット：$Q_{\max}=7$）**：
重みが $w = [0.35,\; -0.70,\; 0.14]$ のとき：
1. 最大絶対値は $0.70$ ⇒ 縮尺 $s = 0.70 / 7 = 0.10$。
2. 割って四捨五入：$[3.5,\; -7.0,\; 1.4] \xrightarrow{\text{round}} q = [4,\; -7,\; 1]$。
3. 逆量子化：$\hat{w} = [0.40,\; -0.70,\; 0.10]$。
4. 誤差 $|w - \hat{w}| = [0.05,\; 0.00,\; 0.04]$（常に半目盛り $s/2 = 0.05$ 以下！）。

**⑥ 外れ値の罠**：もしここに1個だけ `10.0` が混ざって $w = [0.5,\; -0.5,\; 10.0]$ になると、$s = 10.0 / 7 \approx 1.43$ に跳ね上がり、$0.5 / 1.43 \approx 0.35 \to 0$、$-0.5 / 1.43 \approx -0.35 \to 0$ と**普通の重みが全部0に潰れて**しまいます！

**⑦ 理解のための最小Pythonコード**：

```python no-run
ws = [0.35, -0.70, 0.14]
scale = max(abs(x) for x in ws) / 7          # 0.10
q = [round(x / scale) for x in ws]           # [4, -7, 1]
w_hat = [qi * scale for qi in q]             # [0.40, -0.70, 0.10]
errs = [abs(a - b) for a, b in zip(ws, w_hat)]
```

では、PyTorchで8ビットのabsmax量子化を動かして誤差を確かめてみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
torch.manual_seed(0)
w = torch.randn(8) * 0.1
scale = w.abs().max() / 127                     # 最大値を127に合わせる
q = torch.round(w / scale).to(torch.int8)       # 8ビットの整数に
w_back = q.float() * scale                      # 元に戻す（逆量子化）
print("元の重み  ：", w.numpy().round(4))
print("8ビット整数：", q.numpy())
print("戻した重み：", w_back.numpy().round(4))
print("最大の誤差：", (w - w_back).abs().max().item())

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
元の重み  ： [ 0.1541 -0.0293 -0.2179  0.0568 -0.1085 -0.1399  0.0403  0.0838]
8ビット整数： [  90  -17 -127   33  -63  -82   24   49]
戻した重み： [ 0.1544 -0.0292 -0.2179  0.0566 -0.1081 -0.1407  0.0412  0.0841]
最大の誤差： 0.0008392892777919769
```

</details>

> **💡 理論のポイント：量子化の考え方**
>
> - float32（4バイト）の重みを、int8（1バイト）の整数と、1つの「縮尺」（scale）で表す。メモリは約4分の1
> - 戻すと少し誤差が出る（上の例では最大0.0008）。LLMの重みはたくさんあるので、1つ1つの小さな誤差は、全体ではあまり効かないことが多い
> - 4ビット（-8〜7の16段階など）まで粗くすると、メモリは8分の1。誤差は大きくなるが、工夫（重みをグループに分けて別々の縮尺を使う、正規分布に合わせた段階を使う **NF4** など）で、性能の低下を小さく抑えられる
>
> ただし、**外れ値**（とても大きな重み）が1つあると、縮尺が大きくなり、ほかの全部の重みが粗くなってしまいます。実際の量子化手法の工夫の多くは、この外れ値との戦いです。

> **📘 基本情報とのつながり：固定小数点と浮動小数点**
>
> 基本情報で、小数の表し方に「固定小数点」と「浮動小数点」があることを習いましたね。量子化は、ある意味で浮動小数点数を「縮尺つきの固定小数点（整数）」に変換する作業です。ビット数を減らせば、表せる段階の数（8ビットなら256段階、4ビットなら16段階）が減り、誤差が増える。試験で計算した「表現できる範囲と精度」の話が、そのままLLMのメモリ節約の話になっています。

#### 📝 確認問題（`ch19-q10` · Lv.2 · absmax対称量子化と逆量子化誤差の手計算）

重みベクトル $w = [0.35, -0.70, 0.14]$ を **4ビット対称量子化**（整数範囲 $-7 \sim +7$、$Q_{\max} = 7$）します。縮尺 $s = \frac{\max |w_i|}{7}$、量子化整数 $q_i = \text{round}(w_i / s)$、逆量子化値 $\hat{w}_i = q_i \cdot s$ としたとき、3番目の要素 $0.14$ の**逆量子化誤差** $|0.14 - \hat{w}_3|$ はいくつになりますか？

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：手順は①最大絶対値から縮尺 $s = \max|w_i| / Q_{\max}$ を求める、②対象要素を $s$ で割って四捨五入し $q_i$ を得る、③$q_i \cdot s$ で逆量子化して元の値との差の絶対値を取る、の3段階です。
- **ヒント 2**：$w = [0.35, -0.70, 0.14]$ の最大絶対値は $0.70$、$Q_{\max} = 7$ なので縮尺は $s = 0.70 / 7 = 0.10$ になります。この $s$ を使って $0.14$ を量子化・逆量子化し、誤差 $|0.14 - \hat{w}_3|$ を計算しましょう。
- **正解**：`0.04`

**考え方**：$s = 0.10$ より $q_3 = \text{round}(1.4) = 1$、逆量子化値は $\hat{w}_3 = 1 \times 0.10 = 0.10$ となり、誤差は $|0.14 - 0.10| = 0.04$ です。

1. 最大絶対値：$\max(|0.35|, |-0.70|, |0.14|) = 0.70$。
1. 縮尺：$s = 0.70 / 7 = 0.10$。
1. 量子化：$0.14 / 0.10 = 1.4 \xrightarrow{\text{round}} q_3 = 1$。
1. 逆量子化：$\hat{w}_3 = 1 \times 0.10 = 0.10$。
1. 誤差：$|0.14 - 0.10| = 0.04$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/19-lora.html#ex-ch19-q10

</details>

#### 📝 確認問題（`ch19-q11` · Lv.2 · absmax量子化における外れ値（outlier）の問題とNF4/グループ量子化の対策）

absmax量子化で、重みベクトルの中に1つだけ極端に大きな値（外れ値、例：`[0.2, -0.3, 0.1, 100.0]`）が混ざっていると、何が起きて困りますか？

- **a.** 外れ値 `100.0` だけが自動で0へ切り捨てられるため、結果として残りの普通の重みの量子化精度は上がる
- **b.** 大きな数値を収めるためにint8のビット幅が自動拡張され、1要素あたりのメモリが1バイトから8バイトへ増える
- **c.** 縮尺 `scale` が `100.0` に引っ張られて巨大化し、他の普通の重みが0付近の同じ整数へ丸め潰される
- **d.** 極端な値が混ざることで縮尺 `scale` の計算結果が0に丸められ、量子化時にゼロ除算エラーで停止する

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：absmax量子化の縮尺 `scale = max(|w|) / Q_max` が、たった1つの巨大な外れ値によってどう変化するか計算してみましょう。
- **ヒント 2**：`100.0 / 127` は約 `0.79` になります。普通の重み `0.2, -0.3, 0.1` をこの巨大な `scale` で割って四捨五入すると何になるでしょうか？
- **正解**：`c`

**考え方**：`scale ≒ 0.79` になると、`0.2 / 0.79 ≒ 0.25 -> 0`、`-0.3 / 0.79 ≒ -0.38 -> 0` と普通の重みが一斉に0へ潰れてしまいます。これがLLM量子化における最大の敵「外れ値問題」です。

1. 外れ値 `100.0` により `scale = 100.0 / 127 ≒ 0.787` に巨大化。
1. 普通の重み `0.2, -0.3, 0.1` を `scale` で割るとすべて絶対値0.5未満。
1. 四捨五入するとすべて `0` になり、モデルの表現力が大きく損なわれる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/19-lora.html#ex-ch19-q11

</details>

*(この節の追加問題 `ch19-q12`, `ch19-q13` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/19-lora.html で自動採点・復習できます)*

## QLoRA：4ビットの本体 ＋ LoRA

**QLoRA** は、モデル本体を4ビット（NF4）に量子化して凍結し、その上にLoRAの付せんを貼って学習する方法です。本体のメモリはfloat16の約4分の1（float32比で約8分の1）、学習するのは付せんだけ。この組み合わせで、無料のT4でも数Bのモデルを鍛えられます。

次のコードは、Qwen2.5-1.5B-Instructを4ビットで読み込んでLoRA学習する例です。4ビットの読み込みには `bitsandbytes` というライブラリを使い、**GPUが必要**です。

このセルは、メニューの「ランタイム」→「セッションを再起動する」で**GPUのメモリを空にしてから**実行します。上の節のモデルがGPUに残っていると、メモリの使用量を正しく測れないうえ、メモリが足りなくなることもあるからです。セッション切断でファイルが消えていても動くように、データが無ければ自動で再取得してから読み込み直します。

⚠️ **GPU専用セル（T4等）**：ランタイムがGPUであることを確認してから実行してください。

🔄 **実行前注意**：前のセルで読み込んだモデルがGPUメモリに残っていると不足することがあります。必要に応じて「ランタイム」→「セッションを再起動する」を行ってから実行してください。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
!pip install -q bitsandbytes peft trl
import json, os, urllib.request, torch
if not os.path.exists("bongo_sft.jsonl"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/bongo_sft.jsonl",
        "bongo_sft.jsonl",
    )
if not os.path.exists("bongo_sft.jsonl"):
    raise RuntimeError("bongo_sft.jsonl のダウンロードに失敗しました。ネットワーク接続を確認して再実行するか、付録E（エラー辞典）を参照してください。")
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

train_rows = [json.loads(l) for l in open("bongo_sft.jsonl", encoding="utf-8")]
ds = Dataset.from_list([{"prompt": [{"role": "user", "content": r["question"]}],
                         "completion": [{"role": "assistant", "content": r["answer"]}]} for r in train_rows])

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
big_name = "Qwen/Qwen2.5-1.5B-Instruct"
big_tok = AutoTokenizer.from_pretrained(big_name)
big = AutoModelForCausalLM.from_pretrained(big_name, quantization_config=bnb, device_map="auto")
big = prepare_model_for_kbit_training(big)
print(f"GPUメモリ使用量：{torch.cuda.memory_allocated() / 1e9:.2f} GB")

trainer = SFTTrainer(model=big, train_dataset=ds, processing_class=big_tok,
                     peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                            target_modules="all-linear", task_type="CAUSAL_LM"),
                     args=SFTConfig(output_dir="qlora-bongo", num_train_epochs=3, per_device_train_batch_size=4,
                                    learning_rate=2e-4, logging_steps=5, report_to="none", save_strategy="no",
                                    bf16=False, fp16=True, gradient_checkpointing=True))
trainer.train()
print(f"学習中の最大GPUメモリ：{torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
GPUメモリ使用量：1.20 GB
...
学習中の最大GPUメモリ：3.85 GB
```

</details>

1.5Bのモデルを、4GB以下のメモリで学習できました。同じ方法で、7Bのモデル（4ビットで約4〜5GB）もT4で鍛えられます（時間はかかりますが）。

| 方法 | 1.5Bの学習メモリの目安 | 7Bの学習メモリの目安 |
|---|---|---|
| 全パラメータ（float32 ＋ AdamW） | 約24GB | 約112GB |
| LoRA（本体はfloat16） | 約4〜5GB | 約16GB以上 |
| QLoRA（本体は4ビット） | 約3〜4GB | 約6〜10GB |

（使う文章の長さやバッチの大きさで変わります。）

> **¥0 無料枠のコツ：無料GPUで大きなモデルを鍛える作戦**
>
> 1. まず0.5Bで、データと学習の設定を固める（速いので、失敗してもすぐやり直せる）
> 2. うまくいったら、QLoRAで1.5B → 3B → 7Bと大きくする
> 3. 学習したアダプタだけをドライブやHugging Faceに保存する（数十MB）
>
> いきなり7Bで試すと、1回の失敗で何十分も失います。研究でも「小さく試してから大きく」が鉄則です。

> **⚠️ つまずき注意：よくあるつまずき**
>
> - **`bitsandbytes` のエラー（CUDAが見つからない等）**：4ビットの読み込みにはGPUが必要です。ランタイムが「T4 GPU」になっているか確認しましょう。インストールした直後に「入っていない」と言われるときは、「ランタイム」→「セッションを再起動する」をしてから、QLoRAのセルをもう一度実行しましょう。
> - **LoRAで全然変化しない**：学習率が小さすぎる（$10^{-5}$ など）ことが多いです。$10^{-4}$〜$10^{-3}$ にしてみましょう。また、`print_trainable_parameters()` で学習するパラメータが0でないか確認しましょう。
> - **`merge_and_unload` 後に結果が少し変わる**：4ビットのモデルにマージすると、丸めの誤差が出ます。マージは、float16などで読み込み直した本体に対して行いましょう。

#### 📝 確認問題（`ch19-q14` · Lv.2 · QLoRAの動作原理と4ビット本体への直接マージを避ける理由）

QLoRAで学習したLoRAアダプタを本体に `merge_and_unload()` して配布したいとき、実務で守るべき正しい手順はどれですか？

- **a.** 4ビット量子化済みの本体へ直接マージするほうが、再丸め誤差が0に抑えられて最も高い精度を維持できる
- **b.** QLoRAで学習したLoRAアダプタ自体も4ビット整数のため、float16本体とは型が合わずマージできない
- **c.** 4ビット本体へ直接マージせず、元のベースモデルをfloat16等で読み直してからアダプタをマージする
- **d.** QLoRAアダプタを本体へマージした後は語彙IDがずれるため、トークナイザーを新しく学習し直す必要がある

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：QLoRAでは「凍結された本体は4ビット（粗い格子）」「学習したアダプタ $A, B$ はfloat16/bf16（高精度）」という精度差があります。
- **ヒント 2**：高精度なLoRA差分を粗い4ビット本体へそのまま足し込んで再量子化すると何が起きるか、本文の注意点を思い出しましょう。
- **正解**：`c`

**考え方**：QLoRAでは「保存・保持している本体が4ビット（NF4）」、「学習するLoRAアダプタがfloat16/float32」です。マージ時はfloat16等で元の本体を読み直し、高精度のまま $W + \frac{\alpha}{r}BA$ を合体させるのが定石です。

1. QLoRA学習後、`save_pretrained` で高精度のLoRAアダプタだけ保存する。
1. マージが必要な場合は、元のベースモデルを `dtype=torch.float16` 等で読み込む。
1. `PeftModel.from_pretrained(base_fp16, adapter_dir).merge_and_unload()` で誤差なく合体させる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/19-lora.html#ex-ch19-q14

</details>

#### 📝 確認問題（`ch19-q15` · Lv.3 · LoRAと量子化がVRAMのどの部分を削るのかを総合説明する）

全パラメータ学習では約112GBも必要な7Bモデルが、**QLoRA** を使うと無料のT4 GPU（16GB）でも学習できるようになります。**LoRA** と **4ビット量子化** がそれぞれVRAMの「どの部分」をどのように削減しているのか、役割分担を説明してください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：学習時VRAMの3大要素「①モデル本体の重み」「②学習パラメータの勾配」「③Adamオプティマイザの状態」に分解して考えましょう。
- **ヒント 2**：LoRAが本体凍結によってほぼ消せる要素（②と③）と、4ビット量子化が圧縮する残りの大物（①）の役割分担を明確に書きましょう。
- **解答の目安**：学習時のVRAMは大きく①モデル本体の重み、②学習するパラメータの勾配、③Adamオプティマイザの記録（慣性と歩幅）に分かれます。LoRAは本体を凍結して学習対象を全体の1%前後の低ランク行列A・Bだけに絞ることで、②勾配と③Adamの記録をほぼゼロ近くまで削減します。一方、4ビット量子化（NF4）は残った大物である①凍結されたモデル本体の重みをfloat32（4バイト）の約8分の1（float16の約4分の1、7Bで約4〜5GB）まで圧縮します。この2つを組み合わせることで、7Bモデル全体でも6〜10GB程度に収まりT4で学習できるようになります。

**考え方**：LoRAだけでは本体の重み（7Bのfloat16で約14GB）が残りT4ギリギリになり、量子化だけでは勾配とAdam状態が重いままです。両者が別々のボトルネックを解決するからこそQLoRAは劇的に効きます。

1. LoRAの役割：学習パラメータを<1〜2%に削減 → 勾配とAdam状態（全学習メモリの約75%）を激減させる。
1. 4ビット量子化の役割：凍結した本体重み $W$ を4ビット（0.5バイト/個）に圧縮 → 本体の常駐メモリを1/4〜1/8にする。
1. さらに勾配チェックポインティング（第16章）で活性化メモリも抑え、16GBのT4に収める。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/19-lora.html#ex-ch19-q15

</details>

**✅ できたらチェック**

- [ ] 行列のランクと、低ランクの積で数の個数が減ることを確かめた
- [ ] LoRA層を自作し、最初は元の層と同じ出力になることを確かめた
- [ ] 自作LoRAをQwenに取り付け、0.1%のパラメータだけでボンゴ先生化した
- [ ] PEFTとTRLでLoRA学習し、アダプタだけを保存・マージした
- [ ] absmax量子化を手で計算し、誤差を確かめた
- [ ] QLoRAの仕組みと、無料GPUで大きなモデルを鍛える作戦を説明できる

> **📝 まとめ**
>
> - **LoRA**：重みの変化 $\Delta W$ を低ランクの $BA$ で表し、本体は凍結して $A$・$B$ だけ学習する
> - 学習するパラメータが1%以下になり、勾配とAdamのメモリが激減。保存も**アダプタ**だけでよい
> - **量子化**：重みを少ないビット数（8ビット・4ビット）で表してメモリを節約。外れ値が敵
> - **QLoRA** ＝ 4ビットの本体 ＋ LoRA。無料のT4でも数Bのモデルを鍛えられる

**ボンゴ先生**：これで、キミはモデルを好きなように鍛えられるようになった。でも、ちょっと待ってほしい。「鍛えたら良くなった」と、どうやって言い切る？ 何問か試して「なんとなく良くなった」では、研究にならない。次の章は、多くの人がさぼりがちで、でも一番大事な **評価** の話だ。

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/19-lora.html
- **次の章**：[第20章 本当に良くなった？：評価と統計](https://sakunokyumei.github.io/llm-research-textbook/20-evaluation.html)